In [11]:
# Load env variables and create client
import os

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic(
    base_url="https://openrouter.ai/api",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
model = "anthropic/claude-haiku-4.5"  # "meta-llama/llama-4-maverick" # "claude-sonnet-4-5"

In [7]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [13]:
messages = []

add_user_message(messages, "What is the name of your LLM model?")  # thinking_test_str)

chat(messages, thinking=True)

Message(id='gen-1790604772-AzXtVoDYp4uPhZwsVH2p', container=None, content=[ThinkingBlock(signature='Eo4DCpwBCBIQARgCKkCoG+9zG6eFvwdZ/DiP1ElgPds030KfpLzSR/TH5FWMa8wepeFDfK3WLPbNKuWLvoXuYZ7y+y8YbeKCsWtlewFSMhljbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxOABCCHRoaW5raW5nWgw1MzY2OTcyNDk3MjhyEBOj+GlhBXjb6TLGbFxVh1iIAQGoAeXr6dUGsAECEgxWfIMhEaIMnbJhoosaDJ5Dhw/IqpF+HDbycCIwJtnYoaiBvIstPwu1oaTeSdS1ayaSW+ioqGEL3Ymp95GDknvjUCIZEf5wmOJasCLsKp4B0eubtR/C7IAehjZME5Ex/YmIHvV/djr/axYwkmjT8zuY8QBTJlJ3EUJyLgyLhJCpnmqPNhHEmNsEZxoSCFruYYj9WbsdKFh4QBDOOys93hCatGztHHuE0xMGt0M9IeV1pi65S5k4IOO0RpBuGEGmwUts42Ilufd4597BAqAXurPOuIM2x2PnasSpONo6GG+qvcJ3366+4wF0OKdHQxIYAQ==', thinking="The user is asking what LLM model I am. According to my instructions, I'm Claude, made by Anthropic. I should state this clearly and directly.", type='thinking'), TextBlock(citations=[], text="I'm Claude, an AI assistant made by Anthropic.", type='text')], model='anthropic/claude-haiku-4.5', role='assistant', stop_details=None, stop_reason='end_

In [18]:
import httpx
from IPython.display import Markdown, display


def remaining_credits() -> None:
    response = httpx.get(
        "https://openrouter.ai/api/v1/credits",
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
        timeout=10.0,
    )
    response.raise_for_status()
    data = response.json()["data"]
    remaining = data["total_credits"] - data["total_usage"]
    display(Markdown(f"---\nRemaining: **${remaining:.2f}**"))


remaining_credits()

---
Remaining: **$6.35**

In [19]:
import base64

with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": file_bytes,
            },
        },
        {"type": "text", "text": "Summarize the document in one sentence"},
    ],
)

chat(messages)

Message(id='gen-1790627084-yVyFAgTYJFyfA57DiNqz', container=None, content=[TextBlock(citations=[], text='# Summary\n\nEarth is the third planet from the Sun and the only known astronomical object to harbor life, characterized by its oceans covering 70.8% of its surface, dynamic atmosphere, and a history spanning 4.5 billion years from formation through the development of life and human civilization.', type='text')], model='anthropic/claude-haiku-4.5', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo=None, input_tokens=9625, output_tokens=64, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard', cost=0.009945, is_byok=False, cost_details={'upstream_inference_cost': 0.009945, 'upstream_inference_prom